# 03_gold_grid_incident_summary

## Purpose

Create the Gold grid incident summary table.

This notebook aggregates cleaned grid event Silver data into a business-facing Gold output.

## Expected output

`vattenfall_dev.analytics.gold_grid_incident_summary`

## Expected grain

One row per event day, region, and severity band.

## Expected KPIs

- grid event count
- elevated event count
- total incident duration
- average incident duration
- affected asset count

## Main idea

This Gold table summarizes operational incident pressure and severity patterns for reporting.

In [0]:
import sys
import importlib

repo_src_path = "/Workspace/Users/dirella@startsteps.org/vattenfall-week9-capstone-anna/src"

if repo_src_path not in sys.path:
    sys.path.append(repo_src_path)

import transforms.gold_aggregations as gold_aggregations
import metrics.business_metrics as business_metrics
from utils.logging_utils import log_table_operation, log_row_count

importlib.reload(gold_aggregations)
importlib.reload(business_metrics)

In [0]:
catalog = "vattenfall_dev"

source_table = f"{catalog}.refined.silver_grid_events"
target_table = f"{catalog}.analytics.gold_grid_incident_summary"

print("Gold table:", "Grid Incident Summary")
print("Source table:", source_table)
print("Target table:", target_table)
print("Grain: one row per event_day, region, and severity_band")

In [0]:
grid_df = spark.table(source_table)

source_count = grid_df.count()

log_row_count("Source Silver grid event rows", source_count)

display(grid_df.limit(20))

In [0]:
gold_grid_df = (
    gold_aggregations.aggregate_grid_incident_summary(grid_df)
    .transform(business_metrics.add_operational_status)
)

gold_count = gold_grid_df.count()

log_row_count("Gold grid incident summary rows", gold_count)

display(gold_grid_df)

In [0]:
(
    gold_grid_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

log_table_operation(
    source_table=source_table,
    target_table=target_table,
    operation_name="Create Gold grid incident summary"
)

print("Written Gold table:", target_table)

In [0]:
gold_df = spark.table(target_table)

print("Rows in Gold table:", gold_df.count())
print("Columns:", gold_df.columns)

display(gold_df)

In [0]:
print("Operational status distribution:")

gold_df.groupBy("operational_status").count().show()

print("Severity band distribution:")

gold_df.groupBy("severity_band").count().show()

print("Null checks for Gold grain:")

for column_name in ["event_day", "region", "severity_band"]:
    null_count = gold_df.filter(f"{column_name} IS NULL").count()
    print(column_name, "nulls:", null_count)

print("Duplicate grain check:")

duplicate_count = (
    gold_df
    .groupBy("event_day", "region", "severity_band")
    .count()
    .filter("count > 1")
    .count()
)

print("duplicate event_day-region-severity_band rows:", duplicate_count)